In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

In [37]:
path = "../dataset/VN30_dataset_from_2019.csv"
df = pd.read_csv(path)

df["time"] = pd.to_datetime(df["time"], format="mixed", dayfirst=True, errors="coerce")

df_2020 = df[df["time"].dt.year >= 2020]

stocks_returns = (
    df_2020
    .sort_values("time")
    .groupby("time")
    .apply(lambda x: np.average(
        x["return_1d"],
        weights=x["Market Capital (Bn VND)"] / x["Market Capital (Bn VND)"].sum()
    ))
    .dropna()
)

stocks_returns = stocks_returns.to_frame(name="stocks_return") 
stocks_returns["stocks_return"] = stocks_returns["stocks_return"] * 100

datasets = {}
datasets['Stock returns'] = stocks_returns["stocks_return"]

df_vn30 = pd.read_csv("../dataset/VN30_INDEX.csv")
df_vn30["time"] = pd.to_datetime(df_vn30["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn30_2020 = df_vn30[df_vn30["time"].dt.year >= 2020].copy()
# Corrected column name from 'return_1d' to 'return_1_day'
vn30_returns = df_vn30_2020['return_1_day']
datasets['VN30 Index'] = vn30_returns

df_vn = pd.read_csv("../dataset/VN_INDEX.csv")  
df_vn["time"] = pd.to_datetime(df_vn["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn_2020 = df_vn[df_vn["time"].dt.year >= 2020].copy()
# Corrected column name from 'return_1d' to 'return_1_day'
vn_returns = df_vn_2020['return_1_day'] * 100
datasets['VN Index'] = vn_returns

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f"../dataset/{file}")
    # Standardize the date column to 'time'
    if "Date" in df_temp.columns:
        df_temp.rename(columns={"Date": "time"}, inplace=True)
    
    df_temp["time"] = pd.to_datetime(df_temp["time"], format="mixed", dayfirst=False, errors="coerce")
    df_temp.dropna(subset=['time'], inplace=True) # Drop rows where date conversion failed
    df_temp_2020 = df_temp[df_temp["time"].dt.year >= 2020].copy()
    
    # Ensure 'close' column exists before calculating returns
    if 'close' in df_temp_2020.columns:
        df_temp_2020['return_1d'] = np.log(df_temp_2020['close'] / df_temp_2020['close'].shift(1)).fillna(0) * 100
        datasets[file.replace('.csv', '')] = df_temp_2020['return_1d']
        globals()[f"df_{file.replace('.csv', '')}_2020"] = df_temp_2020
    else:
        print(f"Warning: 'close' column not found in {file}. Skipping return calculation.")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28872\3489981194.py:12: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [6]:
data = {
    "Dataset": [
        "Stock returns",
        "VN30 Index",
        "VN Index",
        "DAX_40",
        "EuroNext_100",
        "IBEX_35",
        "KOSPI_index",
        "SMI",
        "snp500",
        "Nikkei_225",
    ],
    "Train": [946, 886, 949, 1066, 769, 962, 979, 1055, 1054, 1021],
    "Val": [184, 279, 209, 276, 154, 159, 161, 168, 229, 189],
    "Test": [368, 334, 341, 186, 615, 421, 331, 287, 225, 255],
    "KSScore": [0.573188, 0.633911, 0.680706, 0.667419, 0.477094, 0.375380, 0.508218, 0.598059, 0.362910, 0.930103],
}

df = pd.DataFrame(data)
print(df)

# DataFrame used by training code
split_df = df.set_index("Dataset")[["Train", "Val", "Test"]]
print("\n" + "=" * 80)
print("KS SPLIT LENGTHS (FOR TRAINING)")
print("=" * 80)
print(split_df)

         Dataset  Train  Val  Test   KSScore
0  Stock returns    946  184   368  0.573188
1     VN30 Index    886  279   334  0.633911
2       VN Index    949  209   341  0.680706
3         DAX_40   1066  276   186  0.667419
4   EuroNext_100    769  154   615  0.477094
5        IBEX_35    962  159   421  0.375380
6    KOSPI_index    979  161   331  0.508218
7            SMI   1055  168   287  0.598059
8         snp500   1054  229   225  0.362910
9     Nikkei_225   1021  189   255  0.930103

KS SPLIT LENGTHS (FOR TRAINING)
               Train  Val  Test
Dataset                        
Stock returns    946  184   368
VN30 Index       886  279   334
VN Index         949  209   341
DAX_40          1066  276   186
EuroNext_100     769  154   615
IBEX_35          962  159   421
KOSPI_index      979  161   331
SMI             1055  168   287
snp500          1054  229   225
Nikkei_225      1021  189   255


In [3]:
student_t_rows = []

for dataset_name, series in returns_by_dataset.items():
    x = series.to_numpy(dtype=float)
    nu, loc, scale = stats.t.fit(x)
    ks_stat, p_value = stats.kstest(x, "t", args=(nu, loc, scale))
    student_t_rows.append(
        {
            "dataset": dataset_name,
            "n_obs": len(x),
            "nu": nu,
            "loc": loc,
            "scale": scale,
            "ks_stat": ks_stat,
            "p_value": p_value,
            "reject_H0_5pct": bool(p_value < 0.05),
        }
    )

student_t_results = pd.DataFrame(student_t_rows).sort_values("p_value", ascending=False)
student_t_results

,dataset,n_obs,nu,loc,scale,ks_stat,p_value,reject_H0_5pct
4,Nikkei_225,1465,4.157325,0.075569,0.986405,0.011515,0.988904,False
5,SMI,1510,3.381797,0.053074,0.627187,0.015056,0.878174,False
6,snp500,1508,2.847071,0.101773,0.762399,0.015576,0.852029,False
0,DAX_40,1528,2.911452,0.090378,0.765298,0.015696,0.839925,False
1,EuroNext_100,1538,2.961028,0.086785,0.701619,0.016930,0.763394,False
3,KOSPI_index,1471,4.416500,0.083457,0.946155,0.017801,0.732739,False
7,VN30_stock_return,1499,2.520291,0.159579,0.771783,0.018489,0.677608,False
8,VN30_INDEX,1499,2.441149,0.153697,0.782911,0.022427,0.431660,False
2,IBEX_35,1542,3.556553,0.096361,0.823466,0.022734,0.396974,False
9,VN_INDEX,1499,2.393271,0.167796,0.717454,0.025509,0.278753,False


In [4]:
nu_table = student_t_results[["dataset", "nu"]].sort_values("nu", ascending=False).reset_index(drop=True)
nu_table

,dataset,nu
0,KOSPI_index,4.416500
1,Nikkei_225,4.157325
2,IBEX_35,3.556553
3,SMI,3.381797
4,EuroNext_100,2.961028
5,DAX_40,2.911452
6,snp500,2.847071
7,VN30_stock_return,2.520291
8,VN30_INDEX,2.441149
9,VN_INDEX,2.393271


In [5]:
risk_rows = []

for dataset_name, series in returns_by_dataset.items():
    x = series.dropna().astype(float)
    if len(x) < 20:
        continue

    mu = float(x.mean())
    sigma = float(x.std(ddof=1))
    skewness = float(stats.skew(x, bias=False))
    kurt = float(stats.kurtosis(x, fisher=True, bias=False))

    var_95 = float(np.percentile(x, 5))
    tail = x[x <= var_95]
    es_95 = float(tail.mean()) if len(tail) > 0 else np.nan

    wealth = (1.0 + x / 100.0).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    risk_rows.append(
        {
            "dataset": dataset_name,
            "mu_mean_return_pct": mu,
            "sigma_volatility_pct": sigma,
            "skewness": skewness,
            "kurtosis": kurt,
            "VaR_95_pct": var_95,
            "ES_CVaR_95_pct": es_95,
            "max_drawdown_pct": max_drawdown_pct,
        }
    )

risk_table_2020_2025 = pd.DataFrame(risk_rows).sort_values("sigma_volatility_pct", ascending=False).reset_index(drop=True)
risk_table_2020_2025

,dataset,mu_mean_return_pct,sigma_volatility_pct,skewness,kurtosis,VaR_95_pct,ES_CVaR_95_pct,max_drawdown_pct
0,Nikkei_225,0.051546,1.396142,-0.381468,10.372172,-2.071117,-3.149270,-31.861626
1,VN30_INDEX,0.055854,1.381058,-0.885225,4.410993,-2.299412,-3.820149,-44.101228
2,VN30_stock_return,0.062050,1.352698,-0.938357,5.138410,-2.244120,-3.749914,-35.790050
3,snp500,0.049792,1.321852,-0.646549,14.686786,-1.855694,-3.217151,-36.102640
4,VN_INDEX,0.041289,1.294794,-1.051791,5.026584,-2.228904,-3.670317,-41.833897
5,KOSPI_index,0.044259,1.292230,-0.377947,6.240601,-1.949015,-3.056479,-36.535497
6,DAX_40,0.040207,1.278866,-0.710076,13.492614,-1.830314,-3.155861,-39.984623
7,IBEX_35,0.038567,1.269732,-1.419691,18.501578,-1.755772,-3.057431,-40.916542
8,EuroNext_100,0.026518,1.170819,-1.267708,14.706120,-1.696015,-2.993292,-39.131405
9,SMI,0.014759,0.977621,-1.095565,12.977040,-1.343354,-2.417961,-28.586877


In [7]:
data = {
    "Dataset": [
        "Stock returns",
        "VN30 Index",
        "VN Index",
        "DAX_40",
        "EuroNext_100",
        "IBEX_35",
        "KOSPI_index",
        "SMI",
        "snp500",
        "Nikkei_225",
    ],
    "Train": [946, 886, 949, 1066, 769, 962, 979, 1055, 1054, 1021],
    "Val": [184, 279, 209, 276, 154, 159, 161, 168, 229, 189],
    "Test": [368, 334, 341, 186, 615, 421, 331, 287, 225, 255],
    "KSScore": [0.573188, 0.633911, 0.680706, 0.667419, 0.477094, 0.375380, 0.508218, 0.598059, 0.362910, 0.930103],
}

df = pd.DataFrame(data)
print(df)

# DataFrame used by training code
split_df = df.set_index("Dataset")[["Train", "Val", "Test"]]
print("\n" + "=" * 80)
print("KS SPLIT LENGTHS (FOR TRAINING)")
print("=" * 80)
print(split_df)

         Dataset  Train  Val  Test   KSScore
0  Stock returns    946  184   368  0.573188
1     VN30 Index    886  279   334  0.633911
2       VN Index    949  209   341  0.680706
3         DAX_40   1066  276   186  0.667419
4   EuroNext_100    769  154   615  0.477094
5        IBEX_35    962  159   421  0.375380
6    KOSPI_index    979  161   331  0.508218
7            SMI   1055  168   287  0.598059
8         snp500   1054  229   225  0.362910
9     Nikkei_225   1021  189   255  0.930103

KS SPLIT LENGTHS (FOR TRAINING)
               Train  Val  Test
Dataset                        
Stock returns    946  184   368
VN30 Index       886  279   334
VN Index         949  209   341
DAX_40          1066  276   186
EuroNext_100     769  154   615
IBEX_35          962  159   421
KOSPI_index      979  161   331
SMI             1055  168   287
snp500          1054  229   225
Nikkei_225      1021  189   255


In [21]:
path = "../dataset/VN30_dataset_from_2019.csv"
df = pd.read_csv(path)

df["time"] = pd.to_datetime(df["time"], format="mixed", dayfirst=True, errors="coerce")


df_2020 = df[df["time"].dt.year >= 2020]


stocks_returns = (
    df_2020
    .sort_values("time")
    .groupby("time")
    .apply(lambda x: np.average(
        x["return_1d"],
        weights=x["Market Capital (Bn VND)"] / x["Market Capital (Bn VND)"].sum()
    ))
    .dropna()
)

stocks_returns = stocks_returns.to_frame(name="stocks_return") 
stocks_returns["stocks_return"] = stocks_returns["stocks_return"] * 100




datasets = {}

datasets['Stock returns'] = stocks_returns["stocks_return"]

df_vn30 = pd.read_csv("../dataset/vn30_index.csv")
df_vn30["time"] = pd.to_datetime(df_vn30["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn30_2020 = df_vn30[df_vn30["time"].dt.year >= 2020].copy()
vn30_returns = df_vn30_2020['return_1_day']
datasets['VN30 Index'] = vn30_returns

df_vn = pd.read_csv("../dataset/vn_index.csv")  
df_vn["time"] = pd.to_datetime(df_vn["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn_2020 = df_vn[df_vn["time"].dt.year >= 2020].copy()
vn_returns = df_vn_2020['return_1_day'] * 100
datasets['VN Index'] = vn_returns

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f"../dataset/{file}")
    df_temp["time"] = pd.to_datetime(df_temp["time"], format="mixed", dayfirst=True, errors="coerce")
    df_temp_2020 = df_temp[df_temp["time"].dt.year >= 2020].copy()
    returns = np.log(df_temp_2020['close'] / df_temp_2020['close'].shift(1)).fillna(0) * 100
    datasets[file.replace('.csv', '')] = returns

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28872\2719845359.py:14: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [42]:
from ipywidgets import interact, Dropdown, IntSlider, SelectionRangeSlider

def get_series(ds):
    return datasets[ds]

def get_time(ds):
    if ds == "Stock returns":
        return stocks_returns.index
    elif ds == "VN30 Index":
        return df_vn30_2020["time"]
    elif ds == "VN Index":
        return df_vn_2020["time"]
    else:
        return globals()[f"df_{ds}_2020"]["time"]

def get_split(ds):
    if ds not in split_df.index:
        return 0, 0, 0
    tr, va, te = split_df.loc[ds]
    return int(tr), int(tr) + int(va), int(tr) + int(va) + int(te)

def calc_vol(x, typ, window):
    if typ == "abs(return)":
        return x.abs()
    if typ == "vol(rolling)":
        return x.rolling(window).std()
    return x.rolling(window).apply(lambda y: (y**2).mean()**0.5, raw=True)

def plot_vol(ds, typ, window, time_range):
    try:
        x = get_series(ds)
        t = get_time(ds)
        # Convert time_range (tuple of indices) to slice
        idx_start, idx_end = time_range
        x = x.iloc[idx_start:idx_end]
        t = t.iloc[idx_start:idx_end] if hasattr(t, 'iloc') else t[idx_start:idx_end]
        v = calc_vol(x, typ, window)
        # For split, use the original split indices but clipped to the selected range
        tr, va, te = get_split(ds)
        tr = max(0, min(tr - idx_start, idx_end - idx_start))
        va = max(0, min(va - idx_start, idx_end - idx_start))
        te = max(0, min(te - idx_start, idx_end - idx_start))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=t[:tr], y=v[:tr], mode="lines", name="train", line=dict(color="#1f77b4")))
        fig.add_trace(go.Scatter(x=t[tr:va], y=v[tr:va], mode="lines", name="val", line=dict(color="#ff7f0e")))
        fig.add_trace(go.Scatter(x=t[va:te], y=v[va:te], mode="lines", name="test", line=dict(color="#2ca02c")))
        fig.update_layout(title=f"Volatility Analysis: {ds} - {typ} - window={window}", height=400)
        fig.show()
    except Exception as e:
        print(f"An error occurred: {e}")

# Get the max length for the slider
_maxlen = max(len(get_series(ds)) for ds in datasets)

interact(
    plot_vol,
    ds=Dropdown(options=list(datasets.keys())),
    typ=Dropdown(options=["abs(return)", "vol(rolling)", "realized_vol"]),
    window=Dropdown(options=[10, 20, 30, 50, 60, 120, 250], value=60),
    time_range=SelectionRangeSlider(
        options=[(str(i), i) for i in range(_maxlen)],
        index=(0, _maxlen-1),
        description='Time Range',
        continuous_update=False
    )
)

interactive(children=(Dropdown(description='ds', options=('Stock returns', 'VN30 Index', 'VN Index', 'DAX_40',…

<function __main__.plot_vol(ds, typ, window, time_range)>